# Addverb cobot — step-by-step walkthrough (`cobot_py`)

Run the cells **one at a time, top to bottom**. Each section depends on the ones before it.

| Step | What it does |
|---|---|
| 0 | Setup: imports, robot IP, optional simulator |
| 1 | Connect to the robot (TCP) |
| 2 | Read state and feedback |
| 3 | Configure safety / payload / gripper (before power on) |
| 4 | **Power on** (BASE → MOTION) |
| 5 | One-shot goals: `move_to`, `move_through` |
| 6 | **Switch controllers** and stream commands: velocity, joint jog, cartesian jog |
| 7 | Gripper open / close |
| 8 | Free drive → teach waypoints → replay |
| 9 | Errors and recovery |
| 10 | **Power off** and disconnect |

### Before you start (real robot)
* `heal_server` is running on the robot PC, and nothing else is connected to it: no ROS 2 launch, no C++ demo, no other notebook or script. **The robot accepts only one connection.** If you get `Errno 104` / `Connection refused`, something else holds the socket. Restart the robot-side controller.
* The workspace is clear and the **physical E-stop is within reach**. The notebook "stop" cells are not a safety device.
* Commands are only sent while a cell runs. Between cells nothing is streamed to a continuous controller, so each streaming cell does the switch, the stream and the stop together.

### No robot? Use the simulator
Set `SIMULATOR = True` in the next cell. The notebook then starts `tools/fake_robot.py` on `127.0.0.1` for you.

## 0. Setup

In [8]:
import logging
import math
import subprocess
import sys
import time
from pathlib import Path

# ---- edit these ------------------------------------------------------------
SIMULATOR = False          # True -> start tools/fake_robot.py and connect to it
ROBOT_IP = "127.0.0.1"  # must match /opt/addverb/cobot_backend/config/ip.csv
ROBOT_PORT = 15263
# ----------------------------------------------------------------------------

# Find cobot_py/ and make `import cobot` work even without `pip install -e .`
COBOT_PY = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cobot").is_dir())
if str(COBOT_PY / "src") not in sys.path:
    sys.path.insert(0, str(COBOT_PY / "src"))

import cobot
from cobot import (
    API, Cobot, ControlInterrupt, FTType, GripperType, RecordedDataEntry, RobotState, SafetyMode,
    CobotError, CommandTimeout, RobotInError, ValidationError,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
print("cobot_py", cobot.__version__, "from", COBOT_PY)

cobot_py 0.1.0 from /home/harshwadibhasme/cobot_ros2a/cobot_py


Optional: start the simulator (it only runs when `SIMULATOR = True`).

In [9]:
sim = None
if SIMULATOR:
    ROBOT_IP = "127.0.0.1"
    sim = subprocess.Popen(
        [sys.executable, str(COBOT_PY / "tools" / "fake_robot.py"), "--port", str(ROBOT_PORT)],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(1.0)
    print("fake robot running, pid", sim.pid)
else:
    print("using the real robot at", ROBOT_IP, ROBOT_PORT)

using the real robot at 127.0.0.1 15263


## 1. Connect

This opens **the** one TCP connection and does the handshake. It retries for a while if the robot is not ready yet.

⚠️ Run this cell **once**. Running it again would try to open a second connection. The guard below stops that.

In [14]:
if "bot" in globals() and bot.is_connected:
    print("already connected, reusing it")
else:
    bot = Cobot(ROBOT_IP, ROBOT_PORT)

print("connected:", bot.is_connected)

INFO cobot.transport: data line established with 127.0.0.1:15263 after 1 init packet(s), 0.0s
INFO cobot.robot: connected to cobot at 127.0.0.1:15263


connected: True


## 2. Read state and feedback

In [11]:
def show(bot):
    fb = bot.feedback()
    print("state       :", bot.state.name)        # BASE / MOTION / ERROR
    print("controller  :", bot.controller.name)
    print("joint pos   :", [f"{v:+.3f}" for v in fb.joint_position], "rad")
    print("joint vel   :", [f"{v:+.3f}" for v in fb.joint_velocity], "rad/s")
    print("joint torque:", [f"{v:+.2f}" for v in fb.joint_torque], "Nm")
    print("EE pose     :", [f"{v:+.3f}" for v in fb.end_effector_position])

show(bot)

state       : BASE
controller  : NONE
joint pos   : ['+0.000', '+0.000', '+0.000', '+0.000', '+0.000', '+0.000'] rad
joint vel   : ['+0.000', '+0.000', '+0.000', '+0.000', '+0.000', '+0.000'] rad/s
joint torque: ['+0.00', '+0.00', '+0.00', '+0.00', '+0.00', '+0.00'] Nm
EE pose     : ['+0.000', '+0.000', '+0.000', '+0.000', '+0.000', '+0.000']


## 3. Configure (before power on)

These setters only **stage** values. They are sent to the robot with the next `power_on()`.
* **Safety mode**: `NOTHING` (0), `NO_HUMAN_SAFETY` (1, the default), or `NO_POS_SAFETY` (2).
* **Payload**: the mass (kg), centre of mass (m) and inertia of whatever is on the flange. A wrong payload makes gravity compensation wrong.
* **Gripper type**: needed only if you use step 7.

In [ ]:
bot.set_safety(SafetyMode.NO_HUMAN_SAFETY)
bot.set_gripper_type(GripperType.NONE)       # e.g. GripperType.DYNAMIXEL / ROBOTIQ / SUCTION ...
bot.set_ft_sensor(FTType.NONE)
bot.set_payload(mass=1.0, com=[0.0, 0.0, 0.05], moi=[0.0] * 6)  # your real tool; mass must be > 0
print("staged; will be sent at power on")

staged; will be sent at power on


## 4. Power on (BASE → MOTION)

The brakes release here. Stand clear.

In [16]:
if bot.state == RobotState.ERROR:
    print("robot is in ERROR, run the recovery cells in step 9 first")
else:
    bot.power_on()
show(bot)

INFO cobot.robot: powered on (state=MOTION)


state       : MOTION
controller  : NONE
joint pos   : ['-0.001', '+0.029', '+0.002', '-0.001', '+0.000', '+0.000'] rad
joint vel   : ['+0.000', '-0.001', '+0.000', '-0.004', '+0.005', '+0.002'] rad/s
joint torque: ['-0.46', '+6.50', '-11.93', '+0.10', '+0.39', '-0.24'] Nm
EE pose     : ['+0.552', '-0.169', '+0.573', '+3.140', '-0.002', '+0.039']


## 5. One-shot goals (position control)

`move_to` / `move_through` carry their controller inside the goal (`LINEAR_VELOCITY` / `MULTI_POINT`), so you **don't** call `switch_controller` first. Each call blocks until the robot reports the motion finished.

Positions are joint angles in **radians**. Local limit checks reject out-of-range targets before anything is sent (`ValidationError`).

In [18]:
# 5a. Move a single joint a little, relative to where the arm is now
start = bot.joint_positions
target = list(start)
target[0] += 0.2                      # joint 1: +0.2 rad (~11 deg)

bot.move_to(target, duration=5.0)     # 5 s trajectory
print("controller used:", bot.controller.name)
show(bot)

controller used: LINEAR_VELOCITY
state       : MOTION
controller  : LINEAR_VELOCITY
joint pos   : ['+0.398', '+0.029', '+0.002', '-0.001', '+0.000', '+0.000'] rad
joint vel   : ['+0.002', '-0.000', '+0.001', '-0.001', '+0.000', '+0.002'] rad/s
joint torque: ['+7.74', '+9.91', '-15.64', '+0.82', '-1.02', '-0.34'] Nm
EE pose     : ['+0.575', '+0.059', '+0.573', '-3.076', '-0.620', '-0.039']


In [19]:
# 5b. Go back to where we started
bot.move_to(start, duration=5.0)
show(bot)

state       : MOTION
controller  : LINEAR_VELOCITY
joint pos   : ['+0.200', '+0.029', '+0.002', '-0.001', '+0.000', '+0.000'] rad
joint vel   : ['-0.001', '-0.001', '+0.001', '+0.001', '+0.000', '+0.001'] rad/s
joint torque: ['-8.21', '+9.76', '-15.80', '+0.58', '-0.82', '-0.29'] Nm
EE pose     : ['+0.575', '-0.055', '+0.573', '-3.125', '-0.313', '-0.039']


In [21]:
# 5c. Several waypoints. `times` are cumulative seconds from the start of the motion
p0 = bot.joint_positions
p1 = [p0[0] + 0.15, p0[1], p0[2], p0[3], p0[4], p0[5]]
p2 = [p0[0] + 0.15, p0[1], p0[2], p0[3], p0[4], p0[5] + 0.3]

bot.move_through([p1, p2, p0], times=[4.0, 8.0, 12.0])
print("controller used:", bot.controller.name)
show(bot)

controller used: MULTI_POINT
state       : MOTION
controller  : MULTI_POINT
joint pos   : ['+0.200', '+0.029', '+0.002', '-0.001', '+0.000', '+0.001'] rad
joint vel   : ['+0.000', '+0.001', '+0.000', '+0.000', '-0.000', '-0.004'] rad/s
joint torque: ['-8.21', '+11.15', '-20.60', '+0.48', '-1.02', '-2.04'] Nm
EE pose     : ['+0.575', '-0.055', '+0.573', '-3.125', '-0.311', '-0.039']


## 6. Switching controllers and streaming commands

Continuous controllers (`EXTERNAL_VELOCITY`, `JOG_JOINT`, `JOG_RPY`, `FREE_DRIVE`, torque) need two things:
1. A **switch**: `bot.switch_controller(API.X)`. This is a handshake with the robot, which answers EXECUTING and then SUCCEEDED.
2. A **command every cycle** after that.

`bot.control_stream(API.X, rate_hz=...)` does both. It switches on entry, gives you a ticker for the loop, and sends a zero command on exit. That is why each cell below switches and streams in one place.

You can see which controller is active with `bot.controller`.

In [22]:
# 6a. Explicit switch: this is what control_stream does internally on entry
print("before:", bot.controller.name)
bot.switch_controller(API.EXTERNAL_VELOCITY)
print("after :", bot.controller.name)

# ...and then you must start sending commands right away
for _ in range(100):                                   # ~1 s at 100 Hz
    bot.set_velocity([0.0, 0.0, 0.0, 0.0, 0.0, 0.05])  # joint 6 at 0.05 rad/s
    time.sleep(0.01)
bot.set_velocity([0.0] * 6)                            # stop
show(bot)

before: MULTI_POINT


INFO cobot.robot: controller active: EXTERNAL_VELOCITY


after : EXTERNAL_VELOCITY
state       : MOTION
controller  : EXTERNAL_VELOCITY
joint pos   : ['+0.199', '+0.029', '+0.002', '-0.001', '+0.000', '+0.049'] rad
joint vel   : ['+0.000', '+0.002', '+0.001', '-0.001', '-0.002', '+0.039'] rad/s
joint torque: ['-8.52', '+12.55', '-21.37', '+0.44', '-1.16', '+3.49'] Nm
EE pose     : ['+0.575', '-0.056', '+0.573', '-3.131', '-0.235', '-0.039']


In [23]:
# 6b. Joint velocity: smooth sinusoid on joint 2 for 5 s (max 0.1 rad/s)
with bot.control_stream(API.EXTERNAL_VELOCITY, rate_hz=100) as stream:
    for t in stream.ticks(seconds=5.0):
        bot.set_velocity([0.0, 0.1 * math.sin(2 * math.pi * 0.2 * t), 0.0, 0.0, 0.0, 0.0])
show(bot)

INFO cobot.robot: controller active: EXTERNAL_VELOCITY


state       : MOTION
controller  : EXTERNAL_VELOCITY
joint pos   : ['+0.199', '+0.029', '+0.002', '-0.001', '+0.000', '+0.050'] rad
joint vel   : ['+0.000', '+0.001', '+0.000', '-0.001', '-0.003', '-0.001'] rad/s
joint torque: ['-7.90', '+7.90', '-22.15', '+0.24', '-1.12', '+1.99'] Nm
EE pose     : ['+0.575', '-0.056', '+0.573', '-3.131', '-0.234', '-0.039']


In [25]:
# 6c. Joint jogging: joint 1 forward for 2 s, then back for 2 s
with bot.control_stream(API.JOG_JOINT, rate_hz=100) as stream:
    for t in stream.ticks(seconds=4.0):
        direction = 1.0 if t < 2.0 else -1.0
        bot.jog_joint([0.2 * direction, 0.0, 0.0, 0.0, 0.0, 0.0])
print("controller:", bot.controller.name)
show(bot)

INFO cobot.robot: controller active: JOG_JOINT


controller: JOG_JOINT
state       : MOTION
controller  : JOG_JOINT
joint pos   : ['+0.199', '+0.029', '+0.002', '-0.001', '+0.000', '+0.050'] rad
joint vel   : ['-0.015', '-0.002', '+0.000', '-0.002', '+0.000', '-0.000'] rad/s
joint torque: ['-7.59', '+9.14', '-20.60', '+0.29', '-1.12', '+1.84'] Nm
EE pose     : ['+0.575', '-0.056', '+0.573', '-3.131', '-0.234', '-0.039']


In [28]:
# 6d. Cartesian jogging (JOG_RPY): [x, y, z, roll, pitch, yaw]; small +z then -z
with bot.control_stream(API.JOG_RPY, rate_hz=100) as stream:
    for t in stream.ticks(seconds=4.0):
        vz = 2 if t < 2.0 else -2
        bot.jog_cartesian([0.0, 0.0, vz, 0.0, 0.0, 0.0])
print("controller:", bot.controller.name)
show(bot)

INFO cobot.robot: controller active: JOG_RPY


controller: JOG_RPY
state       : MOTION
controller  : JOG_RPY
joint pos   : ['+0.199', '+0.028', '+0.000', '-0.005', '+0.000', '+0.050'] rad
joint vel   : ['-0.002', '+0.000', '+0.030', '+0.027', '+0.003', '+0.000'] rad/s
joint torque: ['-5.11', '+34.85', '-10.53', '+7.13', '-0.73', '+1.70'] Nm
EE pose     : ['+0.575', '-0.056', '+0.573', '-3.132', '-0.233', '-0.034']


### In-notebook stop
Run this if a stream was interrupted (for example with **Kernel → Interrupt**) and you want to command zero immediately. It does not replace the E-stop.

In [29]:
bot.send_control(ControlInterrupt(ext_ctrl=[0.0] * 6, joint_jog=[0.0] * 6, cart_jog=[0.0] * 6))
show(bot)

state       : MOTION
controller  : JOG_RPY
joint pos   : ['+0.199', '+0.028', '+0.003', '-0.002', '+0.000', '+0.050'] rad
joint vel   : ['+0.000', '-0.001', '-0.000', '+0.002', '+0.001', '-0.005'] rad/s
joint torque: ['-4.34', '+34.38', '-11.62', '+4.31', '-0.73', '+1.84'] Nm
EE pose     : ['+0.575', '-0.056', '+0.572', '-3.131', '-0.233', '-0.036']


> **Torque control** (`API.EXTERNAL_TORQUE`, `API.GCOMP_EXTERNAL_TORQUE`) is left out on purpose. `EXTERNAL_TORQUE` is a *critical* controller: you must pass `first_command=` with a valid gravity-compensated torque, or the arm drops during the switch. Test it on the simulator first.

## 7. Gripper

The gripper type must have been set in step 3 **before** power on. Skip this section if you have no gripper.

In [ ]:
HAS_GRIPPER = False   # set True once a gripper type was configured in step 3

if HAS_GRIPPER:
    bot.close_gripper(force=40.0)   # clamp force in N (0-150)
    time.sleep(2.0)
    bot.open_gripper()
else:
    print("skipped (HAS_GRIPPER = False)")

## 8. Free drive → teach → replay

1. Switch to `FREE_DRIVE`. The arm becomes compliant and you can push it by hand.
2. Move the arm to a pose, then **run the "record waypoint" cell**. Repeat for each waypoint.
3. Replay the recorded path.

In [30]:
bot.switch_controller(API.FREE_DRIVE)
waypoints = []
SECONDS_BETWEEN_POINTS = 4.0
print("free drive active, guide the arm by hand")

INFO cobot.robot: controller active: FREE_DRIVE


free drive active, guide the arm by hand


In [ ]:
# Run this cell once per waypoint, after you have positioned the arm
waypoints.append(
    RecordedDataEntry(
        desired_pos=bot.joint_positions,
        desired_time=SECONDS_BETWEEN_POINTS * (len(waypoints) + 1),
    )
)
print(f"{len(waypoints)} waypoint(s); last:", [f"{v:+.3f}" for v in waypoints[-1].desired_pos])

In [ ]:
# Replay. Runs on the robot; this cell waits for roughly the path duration
ITERATIONS = 1
bot.replay(waypoints, label="notebook_path", iterations=ITERATIONS)
time.sleep(SECONDS_BETWEEN_POINTS * len(waypoints) * ITERATIONS + 1.0)
show(bot)

## 9. Errors and recovery

If a command raises `RobotInError`, or `bot.state` shows `ERROR` (for example after a collision or a limit violation):
* `clear_error()` takes the robot from ERROR back to BASE. Then power on again (step 4).
* `recover()` does the full recovery: it clears the error, powers on in recovery safety, drives home to all-zeros over 15 s, then stops. **Make sure the arm has a clear path to the zero pose.**

In [31]:
show(bot)
if bot.state == RobotState.ERROR:
    bot.clear_error()
    print("cleared; now BASE, run step 4 to power on again")

state       : MOTION
controller  : FREE_DRIVE
joint pos   : ['+0.109', '+0.028', '+0.019', '-0.146', '+0.000', '+0.050'] rad
joint vel   : ['+0.001', '-0.000', '+0.002', '-0.000', '-0.002', '+0.001'] rad/s
joint torque: ['+0.00', '+22.30', '-23.23', '+3.73', '+0.05', '+0.05'] Nm
EE pose     : ['+0.544', '-0.110', '+0.540', '+3.120', '+0.093', '-0.214']


In [ ]:
# Full recovery (homes the arm to all zeros), only run if you need it
RUN_RECOVERY = False
if RUN_RECOVERY:
    bot.recover()
    show(bot)

## 10. Power off and disconnect

`power_off()` takes the robot back to BASE (brakes on) and **closes the connection**. To continue afterwards, run step 1 again.

In [32]:
if bot.is_connected and bot.state == RobotState.MOTION:
    bot.power_off()
bot.disconnect()
print("connected:", bot.is_connected)

INFO cobot.robot: powered off


connected: False


In [ ]:
# Stop the simulator if we started one
if sim is not None:
    sim.terminate()
    sim.wait(timeout=5)
    print("fake robot stopped")